# 00 - Setup

Runs on your local machine. GPU work runs on Modal (`src/modal_app.py`), not in this notebook.

1. installs the Python packages (`requirements.txt`) into the active environment,
2. pulls the external reference repos (`external/*`, git submodules) if they're missing,
3. creates the local working folders (`data/raw`, `data/processed`, `output`),
4. verifies `modal` is installed and authenticated, and reports what the
   `segmentation-data` Volume holds,
5. sets `DATA_ROOT` / `OUTPUT_ROOT` to the local folders. Inside a Modal container the same
   call returns the mounted Volume instead.

## Setup

In [ ]:
# ============================================================
# SETUP: dependencies + external repos + working folders
# ============================================================
import os, subprocess, pathlib, sys

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

# 0. Repo root -----------------------------------------------------------
root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repo root:", root, "\n")

# 1. Python dependencies -------------------------------------------------
# Local environment only. The Modal image builds its own copy of this same
# requirements.txt (src/modal_app.py).
sh("pip install -q -r requirements.txt")

# 2. External reference repos --------------------------------------------
externals = {
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning": (
        "https://github.com/ZhangHH233/Retinal_OCT_Image_Segmentation_via_Deep_Learning.git",
        "ac6d4c5d263ce03b1bc9235b471a88f9dbb0d994",
    ),
    "external/Public-available-retinal-OCT-datasets": (
        "https://github.com/ZhangHH233/Public-available-retinal-OCT-datasets.git",
        "f50b6a390a1211f06bd34f244c8e53dfc110a266",
    ),
}
for path, (url, commit) in externals.items():
    if any(pathlib.Path(path).glob("*")):
        print(f"OK   {path} (already present)")
        continue
    try:
        sh(f"git submodule update --init -- {path}")
    except subprocess.CalledProcessError:
        sh(f"git clone {url} {path}")
        sh(f"git -C {path} checkout {commit}")

# 3. Working folders (gitignored) ----------------------------------------
for d in ["data/raw", "data/processed", "output"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
    print(f"OK   {d}/")

print("\nSetup complete — run the verification cell below.")

## Verify environment

In [ ]:
import sys, os
print("Python:", sys.version.split()[0])

# No GPU check here: training runs in a Modal container, which requests its own
# GPU (src/modal_app.py, DEFAULT_GPU). torch locally is CPU-only and that is fine.
import numpy, scipy, skimage, sklearn, PIL, h5py, imageio, cv2, SimpleITK, matplotlib
import torch, torchvision, einops, timm, thop, bm3d
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__, "| SimpleITK:", SimpleITK.Version())

print("\nRepo layout:")
for path in [
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning/SOTAS",
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning/Metrics",
    "external/Public-available-retinal-OCT-datasets",
    "data/raw", "data/processed", "output",
]:
    print(f"  {'OK     ' if os.path.isdir(path) else 'MISSING'} {path}")

## Verify Modal, then set `DATA_ROOT` / `OUTPUT_ROOT`

One-time, in a terminal:

```bash
modal setup                             # opens a browser to authenticate
modal volume create segmentation-data   # the CLI needs the Volume to exist before `put`
```

Input data lives on the Volume, not on this machine. Seed it with the denoised sets and the
fold definitions, which are the only things `modal_app.py` reads:

```bash
modal volume put segmentation-data data/processed/duke_dme_denoised /data/processed/duke_dme_denoised
modal volume put segmentation-data data/processed/hc_ms_denoised   /data/processed/hc_ms_denoised
modal volume put segmentation-data data/processed/folds.json       /data/processed/folds.json
```

Pull results back with:

```bash
modal volume get segmentation-data /output ./output
```

In [ ]:
# Locally resolve_roots() returns data/ + output/. Inside a Modal container it
# returns the mounted Volume (/vol/data, /vol/output) instead.
import subprocess

from src.paths import resolve_roots
from src.modal_app import VOLUME_NAME

DATA_ROOT, OUTPUT_ROOT = resolve_roots()
print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

# Fails if no token is configured, or if the Volume has not been created yet:
# modal_app.py's create_if_missing only applies when the app runs, not to the CLI.
probe = subprocess.run(
    ['modal', 'volume', 'ls', VOLUME_NAME, '/data/processed'],
    capture_output=True, text=True,
)
print(f'\nVolume {VOLUME_NAME!r} /data/processed:')
if probe.returncode == 0:
    print(probe.stdout.strip() or '(empty — seed it, see the cell above)')
else:
    error = probe.stderr.strip()
    print(error)
    if 'not found' in error:
        print(f'\n-> modal volume create {VOLUME_NAME}')
    else:
        print('\n-> modal setup')